<a href="https://colab.research.google.com/github/gundago/godiraone.github.io/blob/main/RiskAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#setup

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

file_path = [
    "/content/drive/My Drive/studentProject/Diebold_Risk Assessment.xlsx",
    "/content/drive/My Drive/studentProject/Tano_RiskAssessement.xlsx",
    "/content/drive/My Drive/studentProject/Tech_Master_RiskAssessment.xlsx",
    "/content/drive/My Drive/studentProject/Full_Banking_Tech_Risks_Revised_Rating.xlsx"
]

#Load and combine data

df_list =[pd.read_excel(file) for file in file_path]
df = pd.concat(df_list, ignore_index=True)
df.head()



Mounted at /content/drive


,Generic_Risk_Categories,Risk_Description,Severity,Likelihood,Rating,Current_Controls,Manual/Automated,Impact,Likelihood.1,Rating.1,...,Not Started,Ontrack,Behind,Completed,(Risk) Action Owner,Due Date For Corrective action,status,Guidance Notes,Process Category(LEVEL 1),Generic_Risk _Categories
0,3. Third Party Risk,Vendor Risk and Performance Uncertainty,Moderate,Possible,Medium,Vendor Due Diligence: Conduct a thorough asses...,Control,Moderate,Possible,Medium,...,NaN,Ontrack,NaN,NaN,Mcdonald Masedi,2025-02-28 00:00:00,Active,NaN,NaN,NaN
1,7. Technology Risk,System Integration and Compatibility Issues,Major,Possible,High,Compatibility Testing: Conduct rigorous compat...,Manual,Moderate,Very Likely,Medium,...,Not Started,NaN,NaN,NaN,Southwell Mbongwe,2025-02-28 00:00:00,Active,NaN,NaN,NaN
2,3. Third Party Risk,Vendor Service Quality and Business \nContinui...,Major,Very likely,High,Vendor Assessment: Evaluate the vendor's servi...,Manual,Moderate,Possible,Medium,...,NaN,NaN,NaN,Completed,Mcdonald Masedi,2024-09-20 00:00:00,Closed,NaN,NaN,NaN
3,8. Financial Risk,Cost Overruns and Budgetary Limitations,Major,Very likely,High,Cost-Benefit Analysis: Conduct a thorough cost...,Manual,Moderate,Possible,Medium,...,Not Started,NaN,NaN,NaN,Gladys Mochobane,2025-02-28 00:00:00,Active,NaN,NaN,NaN
4,2.Project Complexity,Project Delays and Coordination Issues\n from ...,Major,Possible,High,Effective Communication: Maintain open communi...,Manual,Moderate,Possible,Medium,...,NaN,Ontrack,NaN,NaN,Tawanda Silo,2025-02-28 00:00:00,Active,NaN,NaN,NaN


In [3]:
!pip install -q sentence-transformers
from  sentence_transformers import SentenceTransformer, util


In [4]:
#clean and filter

df = df[['Risk_Description', 'Severity', 'Likelihood', 'Rating', 'Current_Controls']].dropna()
df = df.drop_duplicates()
df['Risk_Description'] = df['Risk_Description'].str.lower().str.strip()


#
df_embed = df[['Risk_Description', 'Current_Controls']].dropna().drop_duplicates()
df_embed['Risk_Description'] = df_embed['Risk_Description'].str.lower().str.strip()

# load model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

#Encode all historical risk descriptions
risk_embeddings = embedding_model.encode(df_embed['Risk_Description'].tolist(),convert_to_tension=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
#Encode Labels For Severity, Likelihood, Rating

from sklearn.preprocessing import LabelEncoder
encoders = {}
for col in ['Severity', 'Likelihood', 'Rating']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

In [8]:
control_encoder = LabelEncoder()
df['Current_Controls'] = control_encoder.fit_transform(df['Current_Controls'])

In [9]:
# Vectorize Risk_Description using tf-idf
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=1000)
X = vectorizer.fit_transform(df['Risk_Description'])

In [10]:
#Train Classifers
from sklearn.ensemble import RandomForestClassifier

models = {}
for col in ['Severity', 'Likelihood', 'Rating', 'Current_Controls']:
    y = df[col]
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X, y)
    models[col] = model




In [11]:
def suggest_controls_by_similarity(problem_statement, top_k=3):
    input_embedding = embedding_model.encode(problem_statement.lower(), convert_to_tensor=True)
    similarities = util.cos_sim(input_embedding, risk_embeddings)[0]
    top_k_indices = similarities.topk(k=top_k).indices
    top_controls = df_embed.iloc[top_k_indices]['Current_Controls'].tolist()
    return list(dict.fromkeys(top_controls))  # Unique, ordered


In [12]:
def predict_risk(statement):
    x_input = vectorizer.transform([statement.lower()])
    results = {}

    # Predict severity, likelihood, rating
    for col in ['Severity', 'Likelihood', 'Rating']:
        pred = models[col].predict(x_input)[0]
        decoded = encoders[col].inverse_transform([pred])[0]
        results[col] = decoded

    # Predict control using classifier
    control_pred = models['Current_Controls'].predict(x_input)[0]
    predicted_control = control_encoder.inverse_transform([control_pred])[0]

    # Suggest controls using similarity
    similar_controls = suggest_controls_by_similarity(statement, top_k=3)

    results['Control_Predicted_By_Model'] = predicted_control
    results['Control_Suggested_By_Similarity'] = similar_controls

    return results


In [19]:
# example useOur
example = "Our bank's legacy systems are not fully compatible with modern banking technologies, leading to inefficiencies in transaction processing, data synchronization, and customer service. This integration gap risks operational downtime and impacts the customer experience."

print(predict_risk(example))

{'Severity': 'Major', 'Likelihood': 'Possible', 'Rating': 'High', 'Control_Predicted_By_Model': 'Compatibility Testing: Conduct rigorous compatibility testing to ensure the new software integrates seamlessly with existing systems.\nMigration Planning: Develop a detailed migration plan to minimize disruptions.', 'Control_Suggested_By_Similarity': ['Compatibility Testing: Conduct rigorous compatibility testing to ensure the new software integrates seamlessly with existing systems.\nMigration Planning: Develop a detailed migration plan to minimize disruptions.', 'System upgrade planning', '1.multi-factor authentication,strong passwords policies, Customer Education on OTP Usage']}


In [ ]:
Example Problem Statements
#"Unauthorized access due to weak password policies in legacy systems."

#"Critical banking applications are not patched regularly, leading to high vulnerability exposure."

#"Employees are using personal cloud storage to transfer customer data."

#"Inadequate network segmentation allows lateral movement of malware across systems."

#"Third-party vendor failed to meet data protection requirements."

#"Absence of real-time monitoring for privileged user activity."

#"Outdated antivirus software failed to detect ransomware attack."

#"Mobile banking app has not undergone recent security testing."

#"Frequent system downtime due to aging infrastructure."

#"No documented disaster recovery procedures in place for critical systems."

#Our bank's legacy systems are not fully compatible with modern banking technologies, leading to inefficiencies in transaction processing, data synchronization, and customer service. This integration gap risks operational downtime and impacts the customer experience.

